# 01 - EDA e Baselines

Este notebook realiza uma analise exploratoria inicial do dataset de churn e compara baselines simples com Scikit-Learn. Ele usa o mesmo pacote `churn` do projeto para evitar divergencia entre notebook e codigo de producao.

In [ ]:
import hashlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, make_scorer, precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from churn.config import RANDOM_SEED  # noqa: E402
from churn.data import load_churn_csv, split_features_target  # noqa: E402
from churn.features import build_preprocessor  # noqa: E402
from churn.metrics import binary_classification_metrics  # noqa: E402

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "churn.csv"
DATASET_SOURCE = "https://www.kaggle.com/datasets/blastchar/telco-customer-churn"
DATASET_ORIGINAL_FILE = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

## 1. Leitura dos dados

In [ ]:
df = load_churn_csv(DATA_PATH)
df.shape

In [ ]:
df.head()

## 2. Schema, nulos e distribuicao do alvo

In [ ]:
df.dtypes

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
target_distribution = df["Churn"].value_counts(normalize=True).rename("proportion")
target_distribution

In [ ]:
counts = df["Churn"].value_counts().sort_index()
ax = counts.plot(kind="bar", color=["#4C78A8", "#F58518"])
ax.set_title("Distribuicao da Classe Churn")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade")
ax.set_xticklabels(["Nao churn", "Churn"], rotation=0)
plt.tight_layout()

## 3. Data readiness

Avaliamos se o dataset esta pronto para modelagem olhando volume, nulos, duplicados, balanceamento do alvo e cardinalidade das categoricas.

In [ ]:
duplicated_rows = int(df.duplicated().sum())
missing_cells = int(df.isna().sum().sum())
missing_rate = float(missing_cells / df.size)
target_churn_rate = float(df["Churn"].mean())
high_cardinality_columns = [
    column for column in df.select_dtypes(include=["object"]).columns if df[column].nunique() > 50
]

readiness = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "duplicated_rows": duplicated_rows,
    "missing_cells": missing_cells,
    "missing_rate": missing_rate,
    "target_churn_rate": target_churn_rate,
    "high_cardinality_columns": len(high_cardinality_columns),
}
readiness["readiness_status"] = (
    "ready" if missing_rate < 0.05 and target_churn_rate > 0 else "review"
)
readiness

## 4. Baselines Scikit-Learn

Avaliamos um baseline ingenuo (`DummyClassifier`) e uma regressao logistica balanceada. A separacao final usa 80% treino e 20% teste, com estratificacao do alvo. A validacao cruzada roda apenas nos 80% de treino.

In [ ]:
x, y = split_features_target(df)
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_SEED,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
}

rows = []
for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(x_train)),
            ("classifier", estimator),
        ]
    )
    cv_results = cross_validate(pipeline, x_train, y_train, cv=cv, scoring=scoring)
    pipeline.fit(x_train, y_train)

    if hasattr(pipeline.named_steps["classifier"], "predict_proba"):
        y_prob = pipeline.predict_proba(x_test)[:, 1]
    else:
        y_prob = pipeline.predict(x_test)

    test_metrics = binary_classification_metrics(y_test, y_prob)
    test_metrics["pr_auc"] = float(average_precision_score(y_test, y_prob))
    row = {"model": model_name}
    row.update({f"cv_{metric}": float(np.mean(cv_results[f"test_{metric}"])) for metric in scoring})
    row.update({f"test_{metric}": value for metric, value in test_metrics.items()})
    rows.append(row)

results = pd.DataFrame(rows).sort_values("test_f1", ascending=False)
results

## 5. Registro no MLflow

Cada baseline e registrado com parametros, metricas, origem e hash do dataset para permitir revalidacao.

In [ ]:
dataset_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
mlflow.set_experiment("churn_eda_baselines")

for row in results.to_dict(orient="records"):
    model_name = str(row["model"])
    with mlflow.start_run(run_name=f"eda_{model_name}"):
        mlflow.log_params(
            {
                "model_name": model_name,
                "dataset_source": DATASET_SOURCE,
                "dataset_original_file": DATASET_ORIGINAL_FILE,
                "dataset_local_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
                "dataset_sha256": dataset_hash,
                "cv_folds": 5,
                "split": "80_20_stratified",
                "seed": RANDOM_SEED,
            }
        )
        mlflow.log_metrics({key: float(value) for key, value in row.items() if key != "model"})
        mlflow.log_metrics(
            {
                f"data_{key}": float(value)
                for key, value in readiness.items()
                if isinstance(value, int | float)
            }
        )

## 6. Observacoes

- Accuracy pode ser enganosa quando a classe churn e minoritaria.
- Recall, F1 e ROC-AUC sao metricas mais importantes para comparar modelos de churn.
- O baseline logistico balanceado serve como referencia minima para a MLP PyTorch.